[Reference](https://medium.com/@sebuzdugan/how-to-use-qwen3-5-397b-a17b-for-agentic-workflows-without-blowing-up-your-gpu-budget-b9536b35a9a9)

# Simple Python client

In [1]:
import os
import requests

API_URL = "https://your-provider.example.com/v1/chat/completions"
API_KEY = os.environ["QWEN_API_KEY"]
def qwen_chat(messages, images=None, max_tokens=512):
    # messages: [{"role": "user", "content": "..."}, ...]
    # images: provider specific (URLs or base64)
    payload = {
        "model": "Qwen3.5-397B-A17B",
        "messages": messages,
        "max_tokens": max_tokens,
    }
    if images:
        payload["images"] = images  # adjust to provider schema
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }
    resp = requests.post(API_URL, json=payload, headers=headers, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    return data["choices"][0]["message"]["content"]
if __name__ == "__main__":
    messages = [
        {"role": "system", "content": "You are a helpful engineering assistant."},
        {"role": "user", "content": "Explain Mixture-of-Experts for production LLM serving."},
    ]
    print(qwen_chat(messages))

# Self‑hosting: Hugging Face + vLLM

In [2]:
# Install vLLM and dependencies
!pip install "vllm>=0.6.0" transformers accelerate

# Optional: authenticate for private weights
!huggingface-cli login

In [3]:
from vllm import LLM, SamplingParams

MODEL_ID = "Qwen/Qwen3.5-397B-A17B"
llm = LLM(
    model=MODEL_ID,
    tensor_parallel_size=8,  # match your GPU count
    dtype="float16",         # or "auto", FP8 where supported
)
sampling_params = SamplingParams(
    temperature=0.2,
    top_p=0.9,
    max_tokens=512,
)
prompt = "Give me a practical breakdown of how MoE affects serving cost for LLMs."
outputs = llm.generate([prompt], sampling_params)
print(outputs[0].outputs[0].text)